# 🧮 Chapter 6: Matrices Part 2 (Multiplication, Transformations, and Rank)

---

## 6.1 Introduction: Matrices in Action

If Chapter 5 was about the anatomy of matrices (shapes, types, transposes), Chapter 6 is about their **behavior**. 

In data science, we rarely just store data in matrices; we multiply them. 
- In Deep Learning, passing data through a neural network layer is simply a Matrix-Vector multiplication: $y = Wx + b$.
- In Computer Vision, rotating or scaling an image is a Matrix-Matrix multiplication.
- In Dimensionality Reduction, we project high-dimensional data into lower dimensions using matrix multiplication.

This chapter covers the rules of matrix multiplication, the geometric intuition of linear transformations, and the concept of Matrix Rank (which measures the true informational volume of a matrix).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style
%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 6.")

## 6.2 Standard Matrix Multiplication (The Dot Product Method)

Standard matrix multiplication is not element-wise. Instead, it is a collection of **dot products** between the rows of the first matrix and the columns of the second matrix.

**The Golden Rule of Dimensions:**
To multiply Matrix $\mathbf{A}$ by Matrix $\mathbf{B}$ (written as $\mathbf{AB}$), the **inner dimensions must match**.
- If $\mathbf{A}$ is $(M \times N)$ and $\mathbf{B}$ is $(N \times K)$.
- The inner dimensions ($N$ and $N$) match. The multiplication is valid.
- The resulting Matrix $\mathbf{C}$ will take on the **outer dimensions**: $(M \times K)$.

To find the value at row $i$ and column $j$ in the new matrix $\mathbf{C}$, you take the dot product of the $i$-th row of $\mathbf{A}$ and the $j$-th column of $\mathbf{B}$.

In [ ]:
# Matrix A: 3 rows, 4 columns (3x4)
A = np.random.randint(1, 10, size=(3, 4))

# Matrix B: 4 rows, 2 columns (4x2)
B = np.random.randint(1, 10, size=(4, 2))

# Multiply A and B
# In Python, we use the '@' operator for matrix multiplication (np.matmul)
C = A @ B

print(f"Matrix A shape: {A.shape}")
print(A)
print(f"\nMatrix B shape: {B.shape}")
print(B)
print(f"\nResulting Matrix C shape: {C.shape} (Took outer dimensions 3 and 2)")
print(C)

# Let's manually verify the top-left element of C (Row 0 of A dot Col 0 of B)
manual_top_left = np.dot(A[0, :], B[:, 0])
print(f"\nManual dot product of A's first row and B's first column: {manual_top_left}")
print(f"Value in C[0,0]: {C[0,0]}")

## 6.3 Matrix Multiplication is NOT Commutative

In basic arithmetic, $3 \times 4 = 4 \times 3$. This is called the commutative property.
**Matrix multiplication is strictly non-commutative:** $\mathbf{AB} \neq \mathbf{BA}$.

In fact, reversing the order often breaks the dimensional rules entirely, resulting in an error. Even if both matrices are square (e.g., $3 \times 3$), $\mathbf{AB}$ will produce a completely different result than $\mathbf{BA}$.

In [ ]:
# Two 2x2 Square Matrices
S1 = np.array([[1, 2], [3, 4]])
S2 = np.array([[5, 6], [7, 8]])

print("Result of S1 @ S2:")
print(S1 @ S2)

print("\nResult of S2 @ S1:")
print(S2 @ S1)

print("\nNotice the results are entirely different. Order matters immensely in Linear Algebra!")

# Trying to reverse our earlier matrices A (3x4) and B (4x2)
try:
    invalid_mult = B @ A
except ValueError as e:
    print(f"\nError when trying B (4x2) @ A (3x4):\n{e}")
    print("Inner dimensions (2 and 3) do not match!")

## 6.4 Geometric Perspective: Linear Transformations

When you multiply a Vector by a Matrix, you are performing a **Linear Transformation**. 
The matrix acts as a mathematical "machine" that takes the input vector, morphs the space it lives in (rotating, stretching, or shearing it), and outputs a new vector in the transformed space.

Let's visualize this by applying a $2 \times 2$ transformation matrix to a set of vectors that form a square shape.

In [ ]:
# Define a set of vectors that form a 1x1 square
# Each column is a coordinate (x, y)
square = np.array([
    [0, 1, 1, 0, 0],  # X coordinates
    [0, 0, 1, 1, 0]   # Y coordinates
])

# Define a Transformation Matrix (A shear matrix)
T_shear = np.array([
    [1, 1],  # X_new = 1*X + 1*Y (Pushes top of square to the right)
    [0, 1]   # Y_new = 0*X + 1*Y (Y stays the same)
])

# Apply the transformation (Matrix * Set of Vectors)
transformed_square = T_shear @ square

# Visualization
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.plot(square[0, :], square[1, :], 'bo-', linewidth=2, markersize=8)
plt.title('Original Space (Square)')
plt.xlim(-1, 3); plt.ylim(-1, 3)
plt.grid(True)
plt.axhline(0, color='k'); plt.axvline(0, color='k')

plt.subplot(1, 2, 2)
plt.plot(transformed_square[0, :], transformed_square[1, :], 'ro-', linewidth=2, markersize=8)
plt.title('Transformed Space (Sheared)')
plt.xlim(-1, 3); plt.ylim(-1, 3)
plt.grid(True)
plt.axhline(0, color='k'); plt.axvline(0, color='k')

plt.tight_layout()
plt.show()

print("The matrix T_shear geometrically warped the coordinate grid, turning the square into a parallelogram.")

## 6.5 Matrix Rank: The Dimensionality of Information

The **Rank** of a matrix is a single number that tells us the true, non-redundant dimensionality of the information contained within that matrix. It ties back perfectly to Chapter 3 (Linear Independence).

- The rank is the number of **linearly independent columns** (or rows) in the matrix.
- The maximum possible rank of a matrix is the smaller of its two dimensions: $r \le \min(M, N)$.
- If a matrix has the maximum possible rank, it is called **Full Rank**.
- If a matrix has redundant columns/rows, its rank will be lower than the max. This is called **Rank-Deficient** or a **Singular Matrix**.

In Machine Learning, if your feature matrix (dataset) is rank-deficient, it means you have perfectly collinear features (e.g., you included both 'Weight in Kg' and 'Weight in Lbs'). This breaks many algorithms like ordinary linear regression because the matrix becomes impossible to invert.

In [ ]:
# 1. A Full Rank Matrix (3x3)
M_full = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]  # The zero breaks the linear pattern, making it independent
])
rank_full = np.linalg.matrix_rank(M_full)
print("Matrix M_full:\n", M_full)
print(f"Rank of M_full: {rank_full} (Full Rank!)\n")

# 2. A Rank-Deficient Matrix (3x3)
# Let's make column 3 an exact copy of column 1 multiplied by 2
M_deficient = np.array([
    [1, 2, 2],  # Col 3 is 2 * Col 1
    [4, 5, 8],
    [7, 8, 14]
])
rank_def = np.linalg.matrix_rank(M_deficient)
print("Matrix M_deficient:\n", M_deficient)
print(f"Rank of M_deficient: {rank_def} (Rank Deficient!)")
print("Even though it is a 3x3 matrix, it only contains 2 dimensions worth of unique information.")

## 6.6 Chapter Summary

In this chapter, we brought matrices to life through multiplication:
- **Dot Product Multiplication:** $\mathbf{C} = \mathbf{AB}$ requires matching inner dimensions and produces outer dimensions.
- **Non-Commutativity:** Order is everything in linear algebra. $\mathbf{AB} \neq \mathbf{BA}$.
- **Transformations:** Multiplying a matrix by a vector is the mathematical equivalent of warping space (scaling, rotating, shearing).
- **Rank:** The ultimate measure of a matrix's true information content. A rank-deficient matrix contains redundant, collinear data, effectively squashing space into a lower dimension.

Next, we will look at how to "undo" a matrix transformation using the Matrix Inverse.